In [16]:
import os

files = [
    f for f in os.listdir()
    if f.lower().endswith(".xlsx")
    and f.startswith("Ringkasan Broker")
]

print("Jumlah file:", len(files))

for f in sorted(files):
    print(f)

Jumlah file: 29
Ringkasan Broker-20260824.xlsx
Ringkasan Broker-20260826.xlsx
Ringkasan Broker-20260827.xlsx
Ringkasan Broker-20260828.xlsx
Ringkasan Broker-20260831.xlsx
Ringkasan Broker-20260901.xlsx
Ringkasan Broker-20260902.xlsx
Ringkasan Broker-20260903.xlsx
Ringkasan Broker-20260904.xlsx
Ringkasan Broker-20260907.xlsx
Ringkasan Broker-20260908.xlsx
Ringkasan Broker-20260909.xlsx
Ringkasan Broker-20260910.xlsx
Ringkasan Broker-20260911.xlsx
Ringkasan Broker-20260914.xlsx
Ringkasan Broker-20260915.xlsx
Ringkasan Broker-20260916.xlsx
Ringkasan Broker-20260917.xlsx
Ringkasan Broker-20260918.xlsx
Ringkasan Broker-20260921.xlsx
Ringkasan Broker-20260922.xlsx
Ringkasan Broker-20260923.xlsx
Ringkasan Broker-20260924.xlsx
Ringkasan Broker-20260925.xlsx
Ringkasan Broker-20260928.xlsx
Ringkasan Broker-20260929.xlsx
Ringkasan Broker-20260930.xlsx
Ringkasan Broker-20261001.xlsx
Ringkasan Broker-20261002 (4).xlsx


In [17]:
import pandas as pd
import os
import re

# =========================================================
# 1. Ambil semua file Ringkasan Broker
# =========================================================

files = sorted([
    f for f in os.listdir()
    if f.lower().endswith(".xlsx")
    and f.startswith("Ringkasan Broker")
])

print("Jumlah file ditemukan:", len(files))

# =========================================================
# 2. Baca dan gabungkan semua file
# =========================================================

all_data = []

for file in files:
    temp = pd.read_excel(file)

    # Ambil tanggal 8 digit dari nama file
    match = re.search(r"(\d{8})", file)

    if match:
        temp["Date"] = match.group(1)
    else:
        temp["Date"] = None

    all_data.append(temp)

df_all = pd.concat(all_data, ignore_index=True)

# =========================================================
# 3. Cleaning
# =========================================================

df_clean = df_all.copy()

# Rapikan nama kolom
df_clean.columns = df_clean.columns.str.strip()

# Rapikan kolom teks
df_clean["Kode Perusahaan"] = (
    df_clean["Kode Perusahaan"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_clean["Nama Perusahaan"] = (
    df_clean["Nama Perusahaan"]
    .astype(str)
    .str.strip()
)

# Rapikan tanggal
df_clean["Date"] = (
    df_clean["Date"]
    .astype(str)
    .str.extract(r"(\d{8})")[0]
)

# Kolom numerik
numeric_cols = ["Volume", "Nilai", "Frekuensi"]

for col in numeric_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

# Urutkan data
df_clean = df_clean.sort_values(
    ["Date", "Kode Perusahaan"]
).reset_index(drop=True)

# =========================================================
# 4. Pemeriksaan dataset
# =========================================================

print("\n===== DATASET CHECK =====")

print("Jumlah file :", len(files))
print("Jumlah baris:", len(df_clean))
print("Jumlah broker unik:", df_clean["Kode Perusahaan"].nunique())

print("\nTanggal:")
print(sorted(df_clean["Date"].dropna().unique()))

print("\nMissing value:")
print(df_clean.isna().sum())

print("\nDuplikat Date + Kode Broker:")
print(
    df_clean.duplicated(
        subset=["Date", "Kode Perusahaan"]
    ).sum()
)

print("\n5 baris pertama:")
display(df_clean.head())

Jumlah file ditemukan: 29

===== DATASET CHECK =====
Jumlah file : 29
Jumlah baris: 2552
Jumlah broker unik: 88

Tanggal:
['20260824', '20260826', '20260827', '20260828', '20260831', '20260901', '20260902', '20260903', '20260904', '20260907', '20260908', '20260909', '20260910', '20260911', '20260914', '20260915', '20260916', '20260917', '20260918', '20260921', '20260922', '20260923', '20260924', '20260925', '20260928', '20260929', '20260930', '20261001', '20261002']

Missing value:
No                 0
Kode Perusahaan    0
Nama Perusahaan    0
Volume             0
Nilai              0
Frekuensi          0
Date               0
dtype: int64

Duplikat Date + Kode Broker:
0

5 baris pertama:


,No,Kode Perusahaan,Nama Perusahaan,Volume,Nilai,Frekuensi,Date
0,1,AD,Sukadana Prima Sekuritas,4142700,1443620900,267,20260824
1,2,AF,Harita Kencana Sekuritas,6224200,5029717500,463,20260824
2,3,AG,Kiwoom Sekuritas Indonesia,581185700,168137959000,32613,20260824
3,4,AH,Shinhan Sekuritas Indonesia,1409100,407006900,81,20260824
4,5,AI,Kay Hian Sekuritas,310831700,147304751400,14172,20260824


In [18]:
# =========================================================
# TIER 1 — KISI MARKET POSITION
# 29 TRADING DAYS
# =========================================================

# ---------------------------------------------------------
# 1. RANKING & MARKET SHARE HARIAN
# ---------------------------------------------------------

# Ranking berdasarkan nilai transaksi setiap hari
df_clean["Rank_Nilai"] = (
    df_clean.groupby("Date")["Nilai"]
    .rank(method="min", ascending=False)
)

# Total nilai transaksi seluruh broker setiap hari
df_clean["Total_Nilai_Harian"] = (
    df_clean.groupby("Date")["Nilai"]
    .transform("sum")
)

# Market share setiap broker setiap hari
df_clean["Market_Share"] = (
    df_clean["Nilai"]
    / df_clean["Total_Nilai_Harian"]
    * 100
)


# ---------------------------------------------------------
# 2. KISI DAILY DATA
# ---------------------------------------------------------

kisi_daily = (
    df_clean[df_clean["Kode Perusahaan"] == "BQ"]
    [[
        "Date",
        "Kode Perusahaan",
        "Nama Perusahaan",
        "Nilai",
        "Volume",
        "Frekuensi",
        "Rank_Nilai",
        "Market_Share"
    ]]
    .sort_values("Date")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. SUMMARY SEMUA BROKER — 29 TRADING DAYS
# ---------------------------------------------------------

broker_summary_29d = (
    df_clean.groupby(
        ["Kode Perusahaan", "Nama Perusahaan"],
        as_index=False
    )
    .agg(
        Avg_Nilai=("Nilai", "mean"),
        Avg_Volume=("Volume", "mean"),
        Avg_Frekuensi=("Frekuensi", "mean"),
        Avg_Market_Share=("Market_Share", "mean"),
        Avg_Rank=("Rank_Nilai", "mean")
    )
    .sort_values("Avg_Nilai", ascending=False)
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 4. KISI SUMMARY
# ---------------------------------------------------------

kisi_summary = broker_summary_29d[
    broker_summary_29d["Kode Perusahaan"] == "BQ"
].copy()

print("========== KISI SUMMARY — 29 TRADING DAYS ==========")
display(kisi_summary)


# ---------------------------------------------------------
# 5. KISI BEST & WORST PERFORMANCE
# ---------------------------------------------------------

best_rank_row = kisi_daily.loc[
    kisi_daily["Rank_Nilai"].idxmin()
]

worst_rank_row = kisi_daily.loc[
    kisi_daily["Rank_Nilai"].idxmax()
]

highest_ms_row = kisi_daily.loc[
    kisi_daily["Market_Share"].idxmax()
]

lowest_ms_row = kisi_daily.loc[
    kisi_daily["Market_Share"].idxmin()
]

print("========== KISI DAILY EXTREMES ==========")

print(
    f"Best Rank       : #{int(best_rank_row['Rank_Nilai'])} "
    f"on {best_rank_row['Date']}"
)

print(
    f"Worst Rank      : #{int(worst_rank_row['Rank_Nilai'])} "
    f"on {worst_rank_row['Date']}"
)

print(
    f"Highest Mkt Share: {highest_ms_row['Market_Share']:.3f}% "
    f"on {highest_ms_row['Date']}"
)

print(
    f"Lowest Mkt Share : {lowest_ms_row['Market_Share']:.3f}% "
    f"on {lowest_ms_row['Date']}"
)


# ---------------------------------------------------------
# 6. SELECTED COMPETITORS
# ---------------------------------------------------------

selected_codes = ["BQ", "XL", "CC", "YP", "PD", "XC"]

tier1_comparison_29d = broker_summary_29d[
    broker_summary_29d["Kode Perusahaan"].isin(selected_codes)
].copy()


# ---------------------------------------------------------
# 7. VALUE DALAM RP TRILIUN
# ---------------------------------------------------------

tier1_comparison_29d["Avg_Nilai_Trillion"] = (
    tier1_comparison_29d["Avg_Nilai"] / 1e12
)


# ---------------------------------------------------------
# 8. GAP VS KISI
# ---------------------------------------------------------

kisi_baseline = tier1_comparison_29d[
    tier1_comparison_29d["Kode Perusahaan"] == "BQ"
].iloc[0]

tier1_comparison_29d["Value_vs_KISI_x"] = (
    tier1_comparison_29d["Avg_Nilai"]
    / kisi_baseline["Avg_Nilai"]
)

tier1_comparison_29d["Market_Share_Gap_pp"] = (
    tier1_comparison_29d["Avg_Market_Share"]
    - kisi_baseline["Avg_Market_Share"]
)

tier1_comparison_29d["Rank_Gap_vs_KISI"] = (
    tier1_comparison_29d["Avg_Rank"]
    - kisi_baseline["Avg_Rank"]
)


# ---------------------------------------------------------
# 9. TABEL FINAL UNTUK PPT
# ---------------------------------------------------------

tier1_ppt = tier1_comparison_29d[[
    "Kode Perusahaan",
    "Nama Perusahaan",
    "Avg_Nilai_Trillion",
    "Avg_Market_Share",
    "Avg_Rank",
    "Avg_Frekuensi",
    "Value_vs_KISI_x",
    "Market_Share_Gap_pp"
]].sort_values(
    "Avg_Nilai_Trillion",
    ascending=False
)

print("\n========== FINAL TIER 1 TABLE ==========")

display(
    tier1_ppt.style.format({
        "Avg_Nilai_Trillion": "{:.3f}",
        "Avg_Market_Share": "{:.3f}%",
        "Avg_Rank": "{:.1f}",
        "Avg_Frekuensi": "{:,.0f}",
        "Value_vs_KISI_x": "{:.2f}x",
        "Market_Share_Gap_pp": "{:+.3f}"
    })
)

========== KISI SUMMARY — 29 TRADING DAYS ==========


,Kode Perusahaan,Nama Perusahaan,Avg_Nilai,Avg_Volume,Avg_Frekuensi,Avg_Market_Share,Avg_Rank
23,BQ,Korea Investment and Sekuritas Indonesia,2.826132e+11,1.187159e+09,58355.172414,0.934142,24.137931


========== KISI DAILY EXTREMES ==========
Best Rank       : #18 on 20260910
Worst Rank      : #31 on 20260930
Highest Mkt Share: 1.510% on 20260910
Lowest Mkt Share : 0.584% on 20260930

========== FINAL TIER 1 TABLE ==========


,Kode Perusahaan,Nama Perusahaan,Avg_Nilai_Trillion,Avg_Market_Share,Avg_Rank,Avg_Frekuensi,Value_vs_KISI_x,Market_Share_Gap_pp
0,XL,Stockbit Sekuritas Digital,3.789,12.487%,1.4,"1,274,281",13.41x,+11.553
2,CC,MANDIRI SEKURITAS,2.971,9.830%,2.9,"357,155",10.51x,+8.896
5,YP,Mirae Asset Sekuritas Indonesia,1.430,4.725%,5.9,"263,223",5.06x,+3.791
6,PD,Indo Premier Sekuritas,1.234,3.924%,7.6,"173,508",4.37x,+2.990
8,XC,Ajaib Sekuritas Asia,0.908,2.997%,9.6,"238,771",3.21x,+2.063
23,BQ,Korea Investment and Sekuritas Indonesia,0.283,0.934%,24.1,"58,355",1.00x,+0.000


In [19]:
avg_frequency = (
    df.groupby(["Kode Perusahaan", "Nama Perusahaan"])["Frekuensi"]
      .mean()
      .reset_index()
      .rename(columns={"Frekuensi": "Avg_Frekuensi"})
      .sort_values("Avg_Frekuensi", ascending=False)
)

avg_frequency

,Kode Perusahaan,Nama Perusahaan,Avg_Frekuensi
80,XL,Stockbit Sekuritas Digital,861811.0
17,CC,MANDIRI SEKURITAS,281275.0
5,AK,UBS Sekuritas Indonesia,204679.0
84,YP,Mirae Asset Sekuritas Indonesia,184127.0
79,XC,Ajaib Sekuritas Asia,165198.0
...,...,...,...
32,GA,BNC Sekuritas Indonesia,160.0
41,II,Danatama Makmur Sekuritas,104.0
43,IT,INTI TELADAN SEKURITAS,91.0
83,YO,Amantara Sekuritas Indonesia,79.0


In [26]:
# =========================================================
# TIER 1 — AVG RANK UNTUK PPT
# =========================================================

avg_rank_ppt = (
    tier1_comparison_29d[
        [
            "Kode Perusahaan",
            "Nama Perusahaan",
            "Avg_Rank"
        ]
    ]
    .sort_values(
        "Avg_Rank",
        ascending=True
    )
    .reset_index(drop=True)
)

# Ranking kompetitif berdasarkan Avg Rank
avg_rank_ppt["Competitive_Rank"] = (
    avg_rank_ppt["Avg_Rank"]
    .rank(
        method="min",
        ascending=True
    )
    .astype(int)
)

# Urutkan berdasarkan Competitive Rank
avg_rank_ppt = (
    avg_rank_ppt
    .sort_values("Competitive_Rank")
    .reset_index(drop=True)
)

print("========== TIER 1 — AVG RANK ==========")

display(
    avg_rank_ppt.style.format({
        "Avg_Rank": "{:.1f}",
        "Competitive_Rank": "{:d}"
    })
)

========== TIER 1 — AVG RANK ==========


,Kode Perusahaan,Nama Perusahaan,Avg_Rank,Competitive_Rank
0,XL,Stockbit Sekuritas Digital,1.4,1
1,CC,MANDIRI SEKURITAS,2.9,2
2,YP,Mirae Asset Sekuritas Indonesia,5.9,3
3,PD,Indo Premier Sekuritas,7.6,4
4,XC,Ajaib Sekuritas Asia,9.6,5
5,BQ,Korea Investment and Sekuritas Indonesia,24.1,6


In [27]:
# =========================================================
# TIER 1 — AVERAGE VOLUME
# =========================================================

volume_ppt = (
    tier1_comparison_29d[
        [
            "Kode Perusahaan",
            "Nama Perusahaan",
            "Avg_Volume"
        ]
    ]
    .sort_values(
        "Avg_Volume",
        ascending=False
    )
    .reset_index(drop=True)
)

print("========== TIER 1 — AVERAGE VOLUME ==========")

display(
    volume_ppt.style.format({
        "Avg_Volume": "{:,.0f}"
    })
)

========== TIER 1 — AVERAGE VOLUME ==========


,Kode Perusahaan,Nama Perusahaan,Avg_Volume
0,XL,Stockbit Sekuritas Digital,"14,461,342,312"
1,CC,MANDIRI SEKURITAS,"6,108,671,900"
2,YP,Mirae Asset Sekuritas Indonesia,"4,424,441,791"
3,XC,Ajaib Sekuritas Asia,"2,477,573,166"
4,PD,Indo Premier Sekuritas,"2,429,606,624"
5,BQ,Korea Investment and Sekuritas Indonesia,"1,187,159,319"


In [28]:
# ============================================================
# COMPETITIVE RANK — DAILY RANKING BASED ON MARKET SHARE
# ============================================================

import pandas as pd

# Pastikan Date bertipe datetime
df['Date'] = pd.to_datetime(df['Date'])

# Hitung total transaction value per hari
daily_total = (
    df.groupby('Date')['Transaction_Value']
      .sum()
      .reset_index(name='Total_Market_Value')
)

# Gabungkan total market value ke dataset
df_rank = df.merge(daily_total, on='Date', how='left')

# Hitung Market Share masing-masing perusahaan per hari
df_rank['Market_Share'] = (
    df_rank['Transaction_Value'] / df_rank['Total_Market_Value']
) * 100

# Ranking perusahaan setiap hari berdasarkan Market Share
# Rank 1 = Market Share terbesar
df_rank['Daily_Rank'] = (
    df_rank.groupby('Date')['Market_Share']
           .rank(method='min', ascending=False)
)

df_rank.head()

KeyError: 'Date'

In [29]:
print(df.columns.tolist())

['No', 'Kode Perusahaan', 'Nama Perusahaan', 'Volume', 'Nilai', 'Frekuensi']


In [30]:
# ============================================================
# COMPETITIVE RANK
# Ranking berdasarkan Nilai, Volume, dan Frekuensi
# ============================================================

# Pastikan data numerik
df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')
df['Nilai'] = pd.to_numeric(df['Nilai'], errors='coerce')
df['Frekuensi'] = pd.to_numeric(df['Frekuensi'], errors='coerce')

# Ranking masing-masing indikator
# Rank 1 = nilai tertinggi
df['Rank_Nilai'] = df['Nilai'].rank(
    ascending=False,
    method='min'
)

df['Rank_Volume'] = df['Volume'].rank(
    ascending=False,
    method='min'
)

df['Rank_Frekuensi'] = df['Frekuensi'].rank(
    ascending=False,
    method='min'
)

# Average Rank
df['Avg_Rank'] = df[
    ['Rank_Nilai', 'Rank_Volume', 'Rank_Frekuensi']
].mean(axis=1)

# Urutkan dari competitive position terbaik
competitive_rank = df.sort_values(
    'Avg_Rank',
    ascending=True
).reset_index(drop=True)

# Tampilkan hasil
competitive_rank[
    [
        'Kode Perusahaan',
        'Nama Perusahaan',
        'Nilai',
        'Volume',
        'Frekuensi',
        'Rank_Nilai',
        'Rank_Volume',
        'Rank_Frekuensi',
        'Avg_Rank'
    ]
]

,Kode Perusahaan,Nama Perusahaan,Nilai,Volume,Frekuensi,Rank_Nilai,Rank_Volume,Rank_Frekuensi,Avg_Rank
0,XL,Stockbit Sekuritas Digital,2420664239986,16737583022,861811,2.0,5.0,1.0,2.666667
1,AK,UBS Sekuritas Indonesia,2753551848888,19750284110,204679,1.0,4.0,3.0,2.666667
2,CC,MANDIRI SEKURITAS,1934335864396,10691047934,281275,4.0,7.0,2.0,4.333333
3,ZP,Maybank Sekuritas Indonesia,1836562596870,26479721482,115690,5.0,3.0,7.0,5.000000
4,YU,CGS International Sekuritas Indonesia,1969297578442,28079682880,83456,3.0,2.0,11.0,5.333333
...,...,...,...,...,...,...,...,...,...
83,RS,Yulie Sekuritas Indonesia Tbk.,1632580500,3626400,205,83.0,84.0,82.0,83.000000
84,PP,Aldiracita Sekuritas Indonesia,1541722500,582600,190,84.0,87.0,83.0,84.666667
85,IT,INTI TELADAN SEKURITAS,976952500,3375500,91,85.0,85.0,86.0,85.333333
86,YO,Amantara Sekuritas Indonesia,136060000,2160000,79,88.0,86.0,87.0,87.000000


In [31]:
competitive_rank.head(10)

,No,Kode Perusahaan,Nama Perusahaan,Volume,Nilai,Frekuensi,Rank_Nilai,Rank_Volume,Rank_Frekuensi,Avg_Rank
0,81,XL,Stockbit Sekuritas Digital,16737583022,2420664239986,861811,2.0,5.0,1.0,2.666667
1,6,AK,UBS Sekuritas Indonesia,19750284110,2753551848888,204679,1.0,4.0,3.0,2.666667
2,18,CC,MANDIRI SEKURITAS,10691047934,1934335864396,281275,4.0,7.0,2.0,4.333333
3,87,ZP,Maybank Sekuritas Indonesia,26479721482,1836562596870,115690,5.0,3.0,7.0,5.000000
4,86,YU,CGS International Sekuritas Indonesia,28079682880,1969297578442,83456,3.0,2.0,11.0,5.333333
5,51,MG,Semesta Indovest Sekuritas,46584962000,1651438460800,90094,6.0,1.0,10.0,5.666667
6,85,YP,Mirae Asset Sekuritas Indonesia,5799778706,1018579426900,184127,8.0,10.0,4.0,7.333333
7,14,BK,J.P. Morgan Sekuritas Indonesia,7411278680,1046776804128,78164,7.0,8.0,12.0,9.000000
8,20,CP,KB Valbury Sekuritas,6922842175,589531176850,98713,10.0,9.0,9.0,9.333333
9,80,XC,Ajaib Sekuritas Asia,2594737800,562905299000,165198,11.0,21.0,5.0,12.333333


In [32]:
df.loc[
    df["Kode Perusahaan"].eq("BQ"),
    [
        "Kode Perusahaan", "Nama Perusahaan",
        "Rank_Nilai", "Rank_Volume",
        "Rank_Frekuensi", "Avg_Rank"
    ]
]

,Kode Perusahaan,Nama Perusahaan,Rank_Nilai,Rank_Volume,Rank_Frekuensi,Avg_Rank
14,BQ,Korea Investment and Sekuritas Indonesia,29.0,32.0,18.0,26.333333


In [33]:
%pip install google-play-scraper pandas openpyxl

  Obtaining dependency information for google-play-scraper from https://files.pythonhosted.org/packages/33/f7/a23ef3cf8efc9ab3aee565971f59906811e6ce95475314ef7b18d02f30ba/google_play_scraper-1.2.7-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/50.2 kB ? eta -:--:--
     -------- ------------------------------- 10.2/50.2 kB ? eta -:--:--
     ----------------------- -------------- 30.7/50.2 kB 262.6 kB/s eta 0:00:01
     ----------------------- -------------- 30.7/50.2 kB 262.6 kB/s eta 0:00:01
     ----------------------- -------------- 30.7/50.2 kB 262.6 kB/s eta 0:00:01
     ----------------------- -------------- 30.7/50.2 kB 262.6 kB/s eta 0:00:01
     ------------------------------ ------- 41.0/50.2 kB 122.9 kB/s eta 0:00:01
     -------------------------------------- 50.2/50.2 kB 142.4 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [36]:
import re
import pandas as pd

from datetime import datetime
from zoneinfo import ZoneInfo
from google_play_scraper import reviews, Sort
from IPython.display import display

competitors = {
    "Ajaib": "ajaib.co.id",
    "Stockbit": "com.stockbit.android",
    "IPOT": "com.indopremier.ipot",
    "Mirae Asset (Neo HOTS)": "id.co.miraeassetdaewoo",
    "KISI (iKISI)": "com.koreainvestment.indonesia",
    "KISI (KOINS Mobile)": "com.koins.mtsand",
}

# Periode satu tahun sampai waktu kode dijalankan.
# Waktu ulasan dari scraper digunakan sebagaimana dikembalikan.
end_date = datetime.now(ZoneInfo("Asia/Jakarta")).replace(tzinfo=None)
start_date = end_date - pd.DateOffset(years=1)

# Batas pengambilan per aplikasi.
# Naikkan jika status menunjukkan batas tercapai.
MAX_REVIEWS = 20000
PAGE_SIZE = 200

# Tema keluhan berdasarkan kata kunci pada ulasan bintang 1–2.
# Ini identifikasi otomatis awal, perlu dicek dari contoh teks.
complaint_patterns = {
    "Login / account access": (
        r"\blogin\b|\blog in\b|\bmasuk\b|\botp\b|"
        r"\bpassword\b|\bpin\b|\blogout\b"
    ),
    "App stability / speed": (
        r"\blemot\b|\blambat\b|\blag\b|\bhang\b|"
        r"\bcrash\b|\bforce close\b|\berror\b|\bbug\b|"
        r"\bloading\b|\bmacet\b"
    ),
    "Trading / order execution": (
        r"\border\b|\bpesanan\b|\btransaksi\b|"
        r"\bjual\b|\bbeli\b|\btrading\b"
    ),
    "Deposit / withdrawal": (
        r"\bdeposit\b|\bwithdraw\w*\b|\bpenarikan\b|"
        r"\btransfer\b|\bsaldo\b|\brdn\b|\btop up\b"
    ),
    "Customer support": (
        r"\bcs\b|\bcustomer service\b|\bsupport\b|"
        r"\blayanan pelanggan\b|\bkomplain\b"
    ),
    "Charts / market data": (
        r"\bchart\b|\bgrafik\b|\bdata\b|"
        r"\brealtime\b|\breal time\b|\bquote\b"
    ),
}

summary_rows = []
review_frames = []

for competitor, app_id in competitors.items():
    print(f"Collecting: {competitor}")

    collected = []
    seen_ids = set()
    token = None
    status = "Complete to date boundary"

    try:
        while True:
            batch, next_token = reviews(
                app_id,
                lang="id",
                country="id",
                sort=Sort.NEWEST,
                count=PAGE_SIZE,
                continuation_token=token,
            )

            if not batch:
                status = "No more reviews returned"
                break

            batch_dates = []

            for item in batch:
                review_id = item.get("reviewId")
                review_date = item.get("at")

                if review_date is not None:
                    batch_dates.append(pd.Timestamp(review_date))

                if not review_id or review_id in seen_ids:
                    continue

                seen_ids.add(review_id)
                collected.append(item)

            # NEWEST: hentikan setelah seluruh halaman melewati
            # awal periode.
            if batch_dates and max(batch_dates) < start_date:
                break

            if next_token is None:
                status = "No more reviews returned"
                break

            if len(collected) >= MAX_REVIEWS:
                status = "Limit reached — partial sample"
                break

            token = next_token

    except Exception as exc:
        status = f"Extraction error — partial data: {exc}"

    df = pd.DataFrame(collected)

    if not df.empty:
        df["at"] = pd.to_datetime(df["at"], errors="coerce")
        df["score"] = pd.to_numeric(df["score"], errors="coerce")

        df = df.loc[
            df["at"].between(start_date, end_date)
            & df["score"].between(1, 5)
        ].copy()

    n = len(df)

    if n == 0:
        summary_rows.append({
            "App": competitor,
            "Reviews Collected": 0,
            "Avg Review Rating": None,
            "Positive %": None,
            "Neutral %": None,
            "Negative %": None,
            "Top Complaint Theme": "N/A",
            "Theme Mentions": None,
            "Status": status,
        })
        continue

    # Rumus:
    # Persentase kategori = jumlah ulasan kategori / total ulasan * 100
    df["Feedback"] = df["score"].map(
        lambda score: (
            "Positive" if score >= 4
            else "Neutral" if score == 3
            else "Negative"
        )
    )

    positive_pct = df["Feedback"].eq("Positive").mean() * 100
    neutral_pct = df["Feedback"].eq("Neutral").mean() * 100
    negative_pct = df["Feedback"].eq("Negative").mean() * 100

    # Cari tema hanya pada ulasan bintang 1–2.
    negative_text = (
        df.loc[df["Feedback"].eq("Negative"), "content"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    theme_counts = {
        theme: int(
            negative_text.str.contains(
                pattern, regex=True, flags=re.IGNORECASE
            ).sum()
        )
        for theme, pattern in complaint_patterns.items()
    }

    highest_count = max(theme_counts.values(), default=0)

    if highest_count > 0:
        # Jika jumlah sama, tampilkan semua tema yang seri.
        top_theme = " / ".join(
            theme for theme, count in theme_counts.items()
            if count == highest_count
        )
    else:
        top_theme = "No matching keyword theme"

    summary_rows.append({
        "App": competitor,
        "Reviews Collected": n,
        "Avg Review Rating": df["score"].mean(),
        "Positive %": positive_pct,
        "Neutral %": neutral_pct,
        "Negative %": negative_pct,
        "Top Complaint Theme": top_theme,
        "Theme Mentions": highest_count,
        "Status": status,
    })

    df["Competitor"] = competitor

    # Simpan tema per ulasan agar hasil bisa diperiksa.
    df["Complaint Themes"] = df.apply(
        lambda row: "; ".join(
            theme
            for theme, pattern in complaint_patterns.items()
            if row["score"] <= 2
            and re.search(
                pattern,
                str(row.get("content") or ""),
                flags=re.IGNORECASE,
            )
        ),
        axis=1,
    )

    review_frames.append(df)

# Tabel ringkasan
df_summary = pd.DataFrame(summary_rows)

# Teks ulasan untuk pemeriksaan
df_reviews = (
    pd.concat(review_frames, ignore_index=True)
    if review_frames else pd.DataFrame()
)

print(
    f"\nPERIOD: {start_date:%d %B %Y} "
    f"– {end_date:%d %B %Y, %H:%M} WIB"
)
print("RATING-BASED FEEDBACK — GOOGLE PLAY")

display(
    df_summary.style
    .hide(axis="index")
    .format({
        "Reviews Collected": "{:,.0f}",
        "Avg Review Rating": "{:.2f}",
        "Positive %": "{:.1f}%",
        "Neutral %": "{:.1f}%",
        "Negative %": "{:.1f}%",
        "Theme Mentions": "{:,.0f}",
    }, na_rep="N/A")
)

Collecting: Ajaib
Collecting: Stockbit
Collecting: IPOT
Collecting: Mirae Asset (Neo HOTS)
Collecting: KISI (iKISI)
Collecting: KISI (KOINS Mobile)

PERIOD: 06 October 2025 – 06 October 2026, 08:36 WIB
RATING-BASED FEEDBACK — GOOGLE PLAY


C:\Users\Muhammad Indragiri E\AppData\Local\Temp\ipykernel_42768\358865133.py:222: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(review_frames, ignore_index=True)


App,Reviews Collected,Avg Review Rating,Positive %,Neutral %,Negative %,Top Complaint Theme,Theme Mentions,Status
Ajaib,"15,558",4.22,79.4%,2.9%,17.6%,Deposit / withdrawal,701,Complete to date boundary
Stockbit,"5,940",3.91,71.7%,3.8%,24.5%,App stability / speed,347,Complete to date boundary
IPOT,"1,630",3.31,53.4%,8.0%,38.6%,Login / account access,159,Complete to date boundary
Mirae Asset (Neo HOTS),116,1.97,19.8%,6.0%,74.1%,Login / account access,39,Complete to date boundary
KISI (iKISI),131,4.18,76.3%,5.3%,18.3%,Trading / order execution,5,No more reviews returned
KISI (KOINS Mobile),19,2.84,42.1%,10.5%,47.4%,App stability / speed,4,Complete to date boundary


In [37]:
%pip install torch transformers sentencepiece tqdm

  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/fe/54/4e0a71376b2fb15aaf65ed9e6a4c8043d36f554338c6daabc63059a5e157/torch-2.14.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for sentencepiece from https://files.pythonhosted.org/packages/ab/62/9e2569867e3dcff7ad6d89642a9615b9801b5cd698abe7df3b490361f66e/sentencepiece-0.2.2-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for setuptools>=77.0.3 from https://files.pythonhosted.org/packages/95/9c/c510029fc6ef33a6275cd2c5d3cecd6613dfd6aa401d57c54f1c18852ccf/setuptools-84.0.0-py3-none-any.whl.metadata
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1.14.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB 2.0 MB/s eta 0:01:01
   -----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.41 requires requests_mock, which is not installed.
conda-repo-cli 1.0.41 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.41 requires nbformat==5.4.0, but you have nbformat 5.7.0 which is incompatible.
conda-repo-cli 1.0.41 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.41 requires requests==2.28.1, but you have requests 2.31.0 which is incompatible.


In [ ]:
%pip uninstall -y torch
%pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.14.1
Uninstalling torch-2.14.1:
  Successfully uninstalled torch-2.14.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print(torch.rand(2, 3))

D:\Anacondaupdate\envs\sentiment311\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Python: D:\Anacondaupdate\envs\sentiment311\python.exe
PyTorch: 2.14.1+cpu
tensor([[0.3558, 0.5866, 0.8108],
        [0.0481, 0.3664, 0.6864]])


In [2]:
%pip install numpy pandas google-play-scraper transformers sentencepiece tqdm jinja2

  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.6 MB 7.9 MB/s eta 0:00:02
   ------------------------------- -------- 10.0/12.6 MB 20.0 MB/s eta 0:00:01
   ----------------------------------- ---- 11.3/12.6 MB 15.7 MB/s eta 0:00:01
   ------------------------------------- -- 11.8/12.6 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 9.4 MB/s  0:00:01
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------- -------------------------- 3.4/9.9 MB 18.3 MB/s eta 0:00:01
   --------------- ------------------------ 3.9/9.9 

In [3]:
import numpy as np
import pandas as pd
import torch
from transformers import pipeline

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Library siap untuk analisis sentimen.")

D:\Anacondaupdate\envs\sentiment311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.4.6
Pandas: 3.0.6
PyTorch: 2.14.1+cpu
Library siap untuk analisis sentimen.


In [1]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

print(torch.tensor([1.0, 2.0]).numpy())

NumPy: 2.4.6
PyTorch: 2.14.1+cpu
[1. 2.]


In [2]:
import pandas as pd

df_reviews = pd.read_pickle("google_play_reviews_1year.pkl")
df_extraction = pd.read_pickle("google_play_extraction_status.pkl")

competitors = dict(
    df_reviews[["Competitor", "App_ID"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

import re
from transformers import pipeline
from tqdm import tqdm
from IPython.display import display

D:\Anacondaupdate\envs\sentiment311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%pip install --force-reinstall --no-cache-dir "numpy==1.26.4"

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
    --------------------------------------- 0.3/15.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/15.8 MB 7.2 MB/s eta 0:00:02
   ------------------- -------------------- 7.6/15.8 MB 16.8 MB/s eta 0:00:01
   ----------------------------- ---------- 11.5/15.8 MB 18.0 MB/s eta 0:00:01
   ----------------------------- ---------- 11.8/15.8 MB 14.2 MB/s eta 0:00:01
   ------------------------------- -------- 12.6/15.8 MB 12.1 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.8 MB 10.7 MB/s eta 0:00:01
   ---------------------------------- ----- 13.6/15.8 MB 9.6 MB/s eta 0:00:01
   ------------------------------------- -- 14.7/15.8 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------  15.5/15.8 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 7.7 MB/s  0:00:02
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    U

  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
import numpy as np
import pandas as pd
import torch

print("NumPy:", np.__version__)
print("Lokasi NumPy:", np.__file__)
print("Pandas:", pd.__version__)

print(pd.Series(["bagus", None]).fillna(""))
print(torch.tensor([1.0, 2.0]).numpy())

NumPy: 1.26.4
Lokasi NumPy: D:\Anacondaupdate\envs\sentiment311\Lib\site-packages\numpy\__init__.py
Pandas: 3.0.6
0    bagus
1         
dtype: str
[1. 2.]


In [2]:
%pip install --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.3"

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   -----------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
import numpy as np
import pandas as pd
import torch

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print(pd.Series(["bagus", None]).fillna(""))
print(torch.tensor([1.0, 2.0]).numpy())


NumPy: 1.26.4
Pandas: 2.2.3
0    bagus
1         
dtype: object
[1. 2.]


In [3]:
import sys
import subprocess
from pathlib import Path

# Environment sementara untuk membaca pickle Pandas 3
converter_dir = Path.cwd() / "pickle_converter"
converter_python = converter_dir / "Scripts" / "python.exe"

if not converter_python.exists():
    subprocess.check_call([
        sys.executable, "-m", "venv", str(converter_dir)
    ])

subprocess.check_call([
    str(converter_python), "-m", "pip", "install",
    "pandas>=3,<4", "numpy>=2,<3",
    "--disable-pip-version-check"
])

# Konversi tanpa mengubah file pickle asli
conversion_code = """
import pandas as pd
import sys

df = pd.read_pickle(sys.argv[1])
df.to_csv(sys.argv[2], index=False, encoding="utf-8-sig")
print(f"Berhasil konversi {len(df):,} review ke CSV.")
"""

subprocess.check_call([
    str(converter_python),
    "-c", conversion_code,
    str(Path("google_play_reviews_1year.pkl").resolve()),
    str(Path("google_play_reviews_1year.csv").resolve()),
])

0

In [4]:
import pandas as pd

df_reviews = pd.read_csv(
    "google_play_reviews_1year.csv",
    keep_default_na=False
)

print("Jumlah review:", len(df_reviews))
display(df_reviews[["Competitor", "content"]].head())

Jumlah review: 23394


,Competitor,content
0,Ajaib,sebaiknya broker memberikan laporan kepemilika...
1,Ajaib,"kenapa saya kasih ranting 1 ,akun pasword suda..."
2,Ajaib,penarikan saldo.. super paling lama...dari pen...
3,Ajaib,mudah diikuti
4,Ajaib,bagus


In [6]:
df_reviews = pd.read_csv(
    "google_play_reviews_1year.csv",
    keep_default_na=False
)

In [7]:
# =====================================================
# 1. IMPORT LIBRARY
# =====================================================
import re
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from IPython.display import display

# =====================================================
# 2. BACA REVIEW DAN FILTER PERIODE
# =====================================================
df_reviews = pd.read_csv(
    "google_play_reviews_1year.csv",
    keep_default_na=False
)

start_date = pd.Timestamp("2025-10-06 00:00:00")
end_date = pd.Timestamp("2026-10-06 09:39:00")

df_reviews["at"] = pd.to_datetime(
    df_reviews["at"], errors="coerce"
)

df_reviews = df_reviews.loc[
    df_reviews["at"].between(start_date, end_date)
].copy().reset_index(drop=True)

if df_reviews.empty:
    raise ValueError("Tidak ada review dalam periode yang dipilih.")

print("Period: 06 October 2025 – 06 October 2026, 09:39")
print(f"Reviews collected: {len(df_reviews):,}")

# =====================================================
# 3. SIAPKAN MODEL RoBERTa
# =====================================================
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

model = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    framework="pt",
    device=-1,  # CPU
)

print("Model labels:", model.model.config.id2label)

def normalize_label(label):
    label = str(label).strip().lower()

    if label.startswith("label_"):
        label_id = int(label.split("_")[-1])
        label = str(model.model.config.id2label[label_id]).lower()

    label_map = {
        "positive": "Positive",
        "neutral": "Neutral",
        "negative": "Negative",
    }

    if label not in label_map:
        raise ValueError(f"Label tidak dikenali: {label}")

    return label_map[label]

# =====================================================
# 4. ANALISIS TEKS REVIEW
# =====================================================
df_sentiment = df_reviews.copy()

df_sentiment["Review_Text"] = (
    df_sentiment["content"].fillna("").astype(str).str.strip()
)

df_sentiment["Text_Sentiment"] = pd.Series(
    pd.NA, index=df_sentiment.index, dtype="string"
)
df_sentiment["Model_Score"] = float("nan")

valid_text = df_sentiment["Review_Text"].ne("")

# Teks identik diproses sekali, lalu dipetakan ke setiap review.
texts = (
    df_sentiment.loc[valid_text, "Review_Text"]
    .drop_duplicates()
    .tolist()
)

batch_size = 16
lookup = {}

for start in tqdm(
    range(0, len(texts), batch_size),
    desc="RoBERTa sentiment"
):
    batch = texts[start:start + batch_size]

    results = model(
        batch,
        batch_size=batch_size,
        truncation=True,
        max_length=512,
    )

    for text, result in zip(batch, results):
        lookup[text] = (
            normalize_label(result["label"]),
            float(result["score"]),
        )

df_sentiment.loc[valid_text, "Text_Sentiment"] = (
    df_sentiment.loc[valid_text, "Review_Text"]
    .map(lambda text: lookup[text][0])
)

df_sentiment.loc[valid_text, "Model_Score"] = (
    df_sentiment.loc[valid_text, "Review_Text"]
    .map(lambda text: lookup[text][1])
)

# Simpan prediksi sebelum membuat tabel.
df_sentiment.to_csv(
    "google_play_roberta_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# 5. TEMA PADA REVIEW YANG DIPREDIKSI NEGATIF
# =====================================================
# Tema memakai kata kunci, bukan klasifikasi tema oleh RoBERTa.
complaint_patterns = {
    "Login / account access": (
        r"\blogin\b|\blog in\b|\botp\b|\bpassword\b|"
        r"\bpin\b|\blogout\b|\btidak bisa masuk\b"
    ),
    "App stability / speed": (
        r"\blemot\b|\blambat\b|\blag\b|\bhang\b|"
        r"\bcrash\b|\bforce close\b|\berror\b|"
        r"\bbug\b|\bloading\b|\bmacet\b"
    ),
    "Trading / order execution": (
        r"\border\b|\bpesanan\b|\btransaksi\b|"
        r"\bjual\b|\bbeli\b|\btrading\b"
    ),
    "Deposit / withdrawal": (
        r"\bdeposit\b|\bwithdraw\w*\b|\bpenarikan\b|"
        r"\btransfer\b|\bsaldo\b|\brdn\b|\btop up\b"
    ),
    "Customer support": (
        r"\bcs\b|\bcustomer service\b|\bsupport\b|"
        r"\blayanan pelanggan\b|\bkomplain\b"
    ),
    "Charts / market data": (
        r"\bchart\b|\bgrafik\b|\bdata\b|"
        r"\brealtime\b|\breal time\b|\bquote\b"
    ),
}

def identify_themes(text):
    return "; ".join(
        theme
        for theme, pattern in complaint_patterns.items()
        if re.search(pattern, text, flags=re.IGNORECASE)
    )

df_sentiment["Complaint_Themes"] = ""

negative_mask = (
    df_sentiment["Text_Sentiment"].eq("Negative").fillna(False)
)

df_sentiment.loc[negative_mask, "Complaint_Themes"] = (
    df_sentiment.loc[negative_mask, "Review_Text"]
    .map(identify_themes)
)

# =====================================================
# 6. TABEL RINGKASAN PER APLIKASI
# =====================================================
rows = []

for app_name, group in df_sentiment.groupby(
    "Competitor", sort=False
):
    analyzed = group.loc[group["Text_Sentiment"].notna()]
    n = len(analyzed)

    negative_texts = analyzed.loc[
        analyzed["Text_Sentiment"].eq("Negative"),
        "Review_Text"
    ]

    theme_counts = {
        theme: int(
            negative_texts.str.contains(
                pattern, case=False, regex=True, na=False
            ).sum()
        )
        for theme, pattern in complaint_patterns.items()
    }

    highest = max(theme_counts.values(), default=0)

    top_theme = (
        " / ".join(
            theme for theme, count in theme_counts.items()
            if count == highest
        )
        if highest > 0
        else "No matching keyword theme"
    )

    rows.append({
        "App": app_name,
        "Reviews Collected": len(group),
        "Texts Analyzed": n,
        "Avg Review Rating": pd.to_numeric(
            group["score"], errors="coerce"
        ).mean(),
        "Positive %": (
            analyzed["Text_Sentiment"].eq("Positive").mean() * 100
            if n else None
        ),
        "Neutral %": (
            analyzed["Text_Sentiment"].eq("Neutral").mean() * 100
            if n else None
        ),
        "Negative %": (
            analyzed["Text_Sentiment"].eq("Negative").mean() * 100
            if n else None
        ),
        "Top Complaint Theme": top_theme if n else "N/A",
        "Theme Mentions": highest if n else None,
    })

df_text_summary = pd.DataFrame(rows)

display(
    df_text_summary.style
    .hide(axis="index")
    .format({
        "Reviews Collected": "{:,.0f}",
        "Texts Analyzed": "{:,.0f}",
        "Avg Review Rating": "{:.2f}",
        "Positive %": "{:.1f}%",
        "Neutral %": "{:.1f}%",
        "Negative %": "{:.1f}%",
        "Theme Mentions": "{:,.0f}",
    }, na_rep="N/A")
)

# =====================================================
# 7. SIMPAN HASIL DAN TAMPILKAN SAMPEL
# =====================================================
df_sentiment.to_csv(
    "google_play_roberta_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

df_text_summary.to_csv(
    "google_play_roberta_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

samples = []

for _, group in df_sentiment.loc[
    df_sentiment["Text_Sentiment"].notna()
].groupby(["Competitor", "Text_Sentiment"]):
    samples.append(
        group.sample(n=min(3, len(group)), random_state=42)
    )

if samples:
    print("\nSAMPLE REVIEWS — CHECK PREDICTIONS")

    display(
        pd.concat(samples, ignore_index=True)[
            [
                "Competitor",
                "Review_Text",
                "Text_Sentiment",
                "Model_Score",
                "Complaint_Themes",
            ]
        ]
    )

print("\nSELESAI. Hasil detail dan tabel ringkasan tersimpan.")

D:\Anacondaupdate\envs\sentiment311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Period: 06 October 2025 – 06 October 2026, 09:39
Reviews collected: 23,394


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 599.12it/s]


Model labels: {0: 'positive', 1: 'neutral', 2: 'negative'}


RoBERTa sentiment: 100%|██████████| 1273/1273 [13:34<00:00,  1.56it/s]


App,Reviews Collected,Texts Analyzed,Avg Review Rating,Positive %,Neutral %,Negative %,Top Complaint Theme,Theme Mentions
Ajaib,"15,557","15,557",4.22,63.6%,9.5%,26.9%,Deposit / withdrawal,786
Stockbit,"5,940","5,940",3.91,48.8%,11.8%,39.5%,App stability / speed,420
IPOT,"1,631","1,631",3.31,41.3%,8.0%,50.6%,Login / account access,172
Mirae Asset (Neo HOTS),116,116,1.97,11.2%,8.6%,80.2%,Login / account access,36
KISI (iKISI),131,131,4.18,64.9%,11.5%,23.7%,Trading / order execution,5
KISI (KOINS Mobile),19,19,2.84,26.3%,15.8%,57.9%,Trading / order execution,4



SAMPLE REVIEWS — CHECK PREDICTIONS


,Competitor,Review_Text,Text_Sentiment,Model_Score,Complaint_Themes
0,Ajaib,masih nyoba dan belajar,Negative,0.603097,
1,Ajaib,verifikasi gua lamaa amat,Negative,0.981554,
2,Ajaib,perhari ini udah 12 jam layanan penarikan rdn ...,Negative,0.983919,Deposit / withdrawal
3,Ajaib,tolong di permudah prosesnya,Neutral,0.541340,
4,Ajaib,"mudah diikuti "" alim937 """,Neutral,0.993269,
5,Ajaib,aplikasi trading yang ok,Neutral,0.952323,
6,Ajaib,pengalaman mantap dan bagus sekali di Ajaib un...,Positive,0.996814,
7,Ajaib,"Benar benar Ajaib, semenjak download apl ini, ...",Positive,0.967450,
8,Ajaib,sudah lama pake ajaib dan selama ini gak ada m...,Positive,0.993479,
9,IPOT,saya pengguna baru. mau masuk akun gak bisa. p...,Negative,0.997647,



SELESAI. Hasil detail dan tabel ringkasan tersimpan.


In [8]:
print(df_sentiment.columns.tolist())

['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'Competitor', 'App_ID', 'Review_Text', 'Text_Sentiment', 'Model_Score', 'Complaint_Themes']


In [9]:
pd.set_option("display.max_colwidth", None)

display(
    df_sentiment.sample(
        n=min(30, len(df_sentiment)),
        random_state=42
    )
)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,Competitor,App_ID,Review_Text,Text_Sentiment,Model_Score,Complaint_Themes
13103,12a2e86b-019d-4a84-9137-fd757d250f86,Lukman Hakim,https://play-lh.googleusercontent.com/a-/ALV-UjU5x8ezKpeOGJBBd83i5ZJYHdBqWBEdOT82oyHxEyKZZ-4_ATy6,good,5,0,2.88.0,2026-01-16 10:56:02,,,2.88.0,Ajaib,ajaib.co.id,good,Positive,0.742257,
3256,00636e23-86dc-4b1c-bb6a-a544b561f356,nigel hashemian ahmad,https://play-lh.googleusercontent.com/a-/ALV-UjWLY-lLVdS_kLUQfvqh33DFU8Bux6Psd-Ni2VhWJNQoiOFe1MQN,sangat mudah trading saham di ajaib,5,0,2.100.0,2026-06-25 03:53:52,"Makasih ya, Kak! 💙 Semoga pengalaman investasi bersama Ajaib selalu menyenangkan. Kami akan terus berusaha memberikan layanan yang semakin baik.✨ -AJ\n",2026-08-02 05:41:48,2.100.0,Ajaib,ajaib.co.id,sangat mudah trading saham di ajaib,Positive,0.753554,
8081,ba91e21c-c98e-4be6-b156-94ab074d3865,Nunik Silviana,https://play-lh.googleusercontent.com/a/ACg8ocKUwloSSvOxBz__JfivbAUbZi4GKXdcpdXQ3m1bpXX3Q8RqpQ=mo,sangat mudah untuk trader baru seperti saya. mudah melihat chart candle ma macd jual beli fast trade. cara buka akun juga sangat mudah. tidak punya tabungan juga bisa nabung di ajaibnya. sangat rekomended buat ritel unyu2 seperti sayah.,5,0,,2026-05-09 09:38:03,"Terima kasih banyak, Kak! 💙 Senang banget bisa menemani perjalanan investasi Kakak. Semoga selalu lancar dan cuan terus bareng Ajaib! ✨ ~YOH",2026-05-19 09:29:07,,Ajaib,ajaib.co.id,sangat mudah untuk trader baru seperti saya. mudah melihat chart candle ma macd jual beli fast trade. cara buka akun juga sangat mudah. tidak punya tabungan juga bisa nabung di ajaibnya. sangat rekomended buat ritel unyu2 seperti sayah.,Positive,0.995868,
18041,94874256-982b-47cb-b547-6e5e7150a8c3,Reza Pecoc,https://play-lh.googleusercontent.com/a-/ALV-UjVtdedTeY-r5xmdgYBS5QS2yTG_7LaypcOIgl1FSTVlSNIqIb-ONQ,broker andalan retail 👍🏻,5,0,3.20.0,2026-04-17 18:34:03,"Halo kak,\n\nSenangnya baca review positif dari Kakak. Kami akan terus berupaya untuk memberikan pengalaman yang baik untuk Kakak. Jika ada saran silakan dapat kirimkan ke kami melalui email ke support@stockbit.com atau kakak bisa klik menu Chat Support di aplikasi Stockbit.\n\nTerima kasih dan semoga hari kakak menyenangkan! ☺️\n",2026-04-20 16:13:45,3.20.0,Stockbit,com.stockbit.android,broker andalan retail 👍🏻,Neutral,0.514604,
17991,4cfbd75c-0e62-4e99-bddc-38f973e8ffc7,Mendy Bawias,https://play-lh.googleusercontent.com/a-/ALV-UjW-ebxDmZ8d-ZZ-dRupQPZWfxEQgv5D_KdPSfKAMgkyyOuWv4rm,terbaik ui dan ux,5,0,3.19.7,2026-04-17 18:35:35,"Halo kak,\n\nSenang sekali baca review positif dari Kakak. Jangan lupa rekomendasikan ke teman-teman kakak yang lain ya ☺️Jika ada saran, silakan dapat kirimkan ke kami melalui email ke support@stockbit.com atau kakak bisa klik menu Chat Support di aplikasi Stockbit.\n\nSemoga harinya menyenangkan!😆\n",2026-04-20 16:51:41,3.19.7,Stockbit,com.stockbit.android,terbaik ui dan ux,Positive,0.950522,
5584,df0b8679-6c97-491c-ada0-f0ba1660fa99,Aditya Afan,https://play-lh.googleusercontent.com/a-/ALV-UjUzE4PKQ7MOTVAYdasX_rOoVZ939vKNZNLBcP0ZCeXrKRiALjI,bisa belajar investasi,5,0,,2026-05-23 13:11:41,"Hai Kak Afan, terima kasih atas bintang 5-nya! 🙏 Senang mendengar Kakak nyaman berinvestasi bersama Ajaib. Ingat, kalau ada pertanyaan soal produk atau butuh bantuan navigasi, Bantuan Ajaib siap membantu kapan pun ya! Semangat terus! 💪 ~YOH",2026-05-25 11:11:41,,Ajaib,ajaib.co.id,bisa belajar investasi,Neutral,0.833369,
221,5e7c0bb8-7f78-44f7-a162-be891dd7e371,Fenny Irawaty,https://play-lh.googleusercontent.com/a/ACg8ocITixxElQRv08LucQ-V03D9v5shasTLv5Cw5qOj-NBy8EfSeQ=mo,"cocok untuk pemula, mudah di pahami, informasi lengkap",5,0,2.105.3,2026-09-10 10:41:34,"Makasih banyak, Kak! 💫 Kami senang bisa jadi bagian dari perjalanan investasi Kakak. Sukses terus untuk langkah investasinya ya! -GN",2026-09-10 12:05:18,2.105.3,Ajaib,ajaib.co.id,"cocok untuk pemula, mudah di pahami, 

In [10]:
# Hanya review dengan label Positive atau Negative
df_polar = df_sentiment[
    df_sentiment["Text_Sentiment"].isin(["Positive", "Negative"])
].copy()

table = pd.crosstab(
    df_polar["Competitor"],
    df_polar["Text_Sentiment"]
).reindex(columns=["Positive", "Negative"], fill_value=0)

table_pct = (
    table.div(table.sum(axis=1), axis=0) * 100
).round(1)

display(table_pct)

Text_Sentiment,Positive,Negative
Competitor,,
Ajaib,70.3,29.7
IPOT,44.9,55.1
KISI (KOINS Mobile),31.2,68.8
KISI (iKISI),73.3,26.7
Mirae Asset (Neo HOTS),12.3,87.7
Stockbit,55.3,44.7


In [12]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [13]:
print(CHECK_FILE.resolve())

D:\KISI_Competitor_Analysis\RoBERTa_Manual_Check.xlsx


In [14]:
from pathlib import Path
import re
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# =========================================================
# 1. FILE DAN PENGATURAN
# =========================================================
FOLDER = Path(r"D:\KISI_Competitor_Analysis")
INPUT_FILE = FOLDER / "google_play_roberta_detail.csv"

MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"
BATCH_SIZE = 16

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE, keep_default_na=False)

if "Competitor" not in df.columns:
    raise ValueError("Kolom Competitor tidak ditemukan.")

text_column = (
    "Review_Text" if "Review_Text" in df.columns else "content"
)

if text_column not in df.columns:
    raise ValueError("Kolom Review_Text atau content tidak ditemukan.")

df["Review_Text"] = df[text_column].astype(str).str.strip()

# Simpan hasil sebelumnya untuk dibandingkan
if "Text_Sentiment" in df.columns:
    df["Previous_Sentiment"] = df["Text_Sentiment"]

# =========================================================
# 2. NORMALISASI EJAAN
#    Teks asli tetap disimpan.
#    Negasi seperti "tidak", "gak", dan "belum" tidak dihapus.
# =========================================================
REPLACEMENTS = {
    "mantaf": "mantap",
    "mantaap": "mantap",
    "mantappp": "mantap",
    "mantapppp": "mantap",
    "treder": "trader",
    "pasword": "password",
    "witdhraw": "withdraw",
    "withdrawl": "withdraw",
    "gabisa": "tidak bisa",
    "gak": "tidak",
    "nggak": "tidak",
    "gk": "tidak",
    "ga": "tidak",
    "tp": "tapi",
    "tdk": "tidak",
    "udh": "sudah",
    "sdh": "sudah",
    "sy": "saya",
    "apk": "aplikasi",
    "tntng": "tentang",
}

def normalize_text(text):
    text = str(text).lower().strip()

    # Batasi pengulangan berlebihan: baguuuus → baguus.
    # Hindari menghapus semua pengulangan huruf.
    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)

    # Beberapa bentuk pujian informal yang umum
    text = re.sub(r"\bmanta+p+\b", "mantap", text)
    text = re.sub(r"\bbagu+s+\b", "bagus", text)

    for original, replacement in REPLACEMENTS.items():
        text = re.sub(
            rf"\b{re.escape(original)}\b",
            replacement,
            text,
        )

    return re.sub(r"\s+", " ", text).strip()

df["Normalized_Text"] = df["Review_Text"].map(normalize_text)

print(f"Jumlah review: {len(df):,}")
print(
    "Teks yang berubah:",
    df["Review_Text"].str.lower().ne(df["Normalized_Text"]).sum(),
)

# =========================================================
# 3. MUAT MODEL ROBERTA
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.to("cpu")
model.eval()

labels = {
    int(key): str(value).lower()
    for key, value in model.config.id2label.items()
}

if set(labels.values()) != {"positive", "neutral", "negative"}:
    raise ValueError(f"Label model tidak sesuai: {labels}")

print("Model labels:", labels)

# =========================================================
# 4. ANALISIS SENTIMEN
#    Analisis teks unik, lalu kembalikan ke seluruh review.
#    Tidak memakai rating bintang sebagai input.
# =========================================================
unique_texts = (
    df.loc[df["Normalized_Text"].ne(""), "Normalized_Text"]
    .drop_duplicates()
    .tolist()
)

prediction_map = {}

for start in tqdm(
    range(0, len(unique_texts), BATCH_SIZE),
    desc="Analyzing sentiment",
):
    batch = unique_texts[start:start + BATCH_SIZE]

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    with torch.inference_mode():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)

    # tolist() tidak memerlukan konversi tensor ke NumPy
    for text, probs in zip(batch, probabilities.tolist()):
        best_id = max(range(len(probs)), key=lambda i: probs[i])

        prediction_map[text] = {
            "Sentiment": labels[best_id].capitalize(),
            "Score": probs[best_id],
        }

df["Text_Sentiment"] = df["Normalized_Text"].map(
    lambda text: prediction_map[text]["Sentiment"]
    if text in prediction_map else "Not analyzed"
)

df["Model_Score"] = df["Normalized_Text"].map(
    lambda text: prediction_map[text]["Score"]
    if text in prediction_map else None
)

# Simpan segera agar hasil analisis tidak hilang
df.to_csv(
    FOLDER / "google_play_roberta_normalized_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

# =========================================================
# 5. TEMA KELUHAN BERDASARKAN KATA KUNCI
#    Hanya diterapkan pada review yang diprediksi Negative.
# =========================================================
THEMES = {
    "Login / account access": [
        r"\blog[\s-]?in\b", r"\bpassword\b",
        r"\bsandi\b", r"\botp\b", r"\bmasuk akun\b",
    ],
    "Registration / verification": [
        r"\bverifikasi\w*\b", r"\bverif\b",
        r"\bregistrasi\b", r"\bpendaftaran\b",
        r"\bdaftar\b", r"\bbuka akun\b", r"\bktp\b",
    ],
    "Deposit / withdrawal": [
        r"\bdeposit\b", r"\btop[\s-]?up\b",
        r"\bwithdraw\w*\b", r"\bwd\b",
        r"\bpenarikan\b", r"\btarik (?:saldo|dana|uang)\b",
        r"\btransfer\b", r"\bsetor\w*\b",
    ],
    "App stability / speed": [
        r"\bbug\w*\b", r"\ber+r?o+r+\w*\b",
        r"\bcrash\w*\b", r"\blemot\b", r"\blelet\b",
        r"\blambat\b", r"\blag\b", r"\bnge(?:lag|leg)\w*\b",
        r"\bloading\b", r"\bhang\b", r"\bforce close\b",
        r"\btidak\s+(?:bisa\s+)?(?:di\s*)?buka\b",
    ],
    "Trading / order execution": [
        r"\border\b", r"\bauto[\s-]?order\b",
        r"\bstop[\s-]?loss\b", r"\bjual\b", r"\bbeli\b",
        r"\beksekusi\b", r"\bmatched\b", r"\breject\w*\b",
    ],
    "Charts / market data": [
        r"\bchart\w*\b", r"\bgrafik\b", r"\bcandle\w*\b",
        r"\border[\s-]?book\b", r"\bdata saham\b",
        r"\bdata harga\b", r"\breal[\s-]?time\b",
    ],
    "Fees / charges": [
        r"\bfee\b", r"\bbiaya\b", r"\bkomisi\b",
        r"\bpotongan\b", r"\btarif\b",
    ],
    "Customer support": [
        r"\bcs\b", r"\bcustomer (?:service|support)\b",
        r"\blayanan pelanggan\b", r"\blive chat\b",
        r"\bchat bantuan\b", r"\badmin\b",
        r"\brespon\w*\b",
    ],
}

def detect_themes(text):
    matches = [
        theme
        for theme, patterns in THEMES.items()
        if any(re.search(pattern, text) for pattern in patterns)
    ]
    return "; ".join(matches) if matches else "Other / unclear"

df["Complaint_Themes"] = [
    detect_themes(text) if sentiment == "Negative" else ""
    for text, sentiment in zip(
        df["Normalized_Text"], df["Text_Sentiment"]
    )
]

# =========================================================
# 6. TABEL SENTIMEN
# =========================================================
analyzed = df[df["Text_Sentiment"].ne("Not analyzed")].copy()
label_order = ["Positive", "Neutral", "Negative"]

counts = pd.crosstab(
    analyzed["Competitor"], analyzed["Text_Sentiment"]
).reindex(columns=label_order, fill_value=0)

summary = counts.copy()
summary["Texts Analyzed"] = counts.sum(axis=1)

for label in label_order:
    summary[f"{label} %"] = (
        counts[label] / summary["Texts Analyzed"] * 100
    ).round(1)

summary["Reviews Collected"] = (
    df.groupby("Competitor").size().reindex(summary.index)
)

if "score" in df.columns:
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    summary["Avg Review Rating"] = (
        df.groupby("Competitor")["score"].mean().round(2)
    )

# =========================================================
# 7. HITUNG THEME MENTIONS
#    Satu review dihitung sekali untuk setiap tema.
# =========================================================
negative = analyzed[
    analyzed["Text_Sentiment"].eq("Negative")
].copy()

theme_rows = negative.assign(
    Theme=negative["Complaint_Themes"].str.split(";")
).explode("Theme")

theme_rows["Theme"] = theme_rows["Theme"].str.strip()
theme_rows = (
    theme_rows.reset_index(names="Review_Row")
    .drop_duplicates(["Review_Row", "Theme"])
)

theme_counts = (
    theme_rows.groupby(["Competitor", "Theme"])
    .size()
    .reset_index(name="Theme Mentions")
)

summary["Top Complaint Theme"] = "No specific theme detected"
summary["Theme Mentions"] = 0

for competitor in summary.index:
    specific = theme_counts[
        theme_counts["Competitor"].eq(competitor)
        & theme_counts["Theme"].ne("Other / unclear")
    ]

    if not specific.empty:
        maximum = specific["Theme Mentions"].max()
        top = specific[specific["Theme Mentions"].eq(maximum)]

        summary.loc[competitor, "Top Complaint Theme"] = (
            "; ".join(sorted(top["Theme"].tolist()))
        )
        summary.loc[competitor, "Theme Mentions"] = int(maximum)

summary["Status"] = "Normalized RoBERTa — not manually validated"

summary = summary.reset_index()
summary.columns.name = None

ppt_columns = [
    "Competitor", "Reviews Collected", "Texts Analyzed",
]

if "Avg Review Rating" in summary.columns:
    ppt_columns.append("Avg Review Rating")

ppt_columns += [
    "Positive %", "Neutral %", "Negative %",
    "Top Complaint Theme", "Theme Mentions", "Status",
]

ppt_table = summary[ppt_columns]

# =========================================================
# 8. TAMPILKAN DAN SIMPAN
# =========================================================
display(ppt_table)

df.to_csv(
    FOLDER / "google_play_roberta_normalized_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

ppt_table.to_csv(
    FOLDER / "google_play_roberta_normalized_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Sampel pemeriksaan: maksimal 10 review per label per aplikasi
sample_parts = [
    group.sample(n=min(10, len(group)), random_state=42)
    for _, group in analyzed.groupby(
        ["Competitor", "Text_Sentiment"]
    )
]

validation_sample = (
    pd.concat(sample_parts, ignore_index=True)
    if sample_parts else analyzed.copy()
)

validation_sample["Manual_Sentiment"] = ""
validation_sample["Validation_Notes"] = ""

# Nama baru setiap ekspor agar tidak menimpa pemeriksaan lama
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_file = FOLDER / f"KISI_RoBERTa_Normalized_{timestamp}.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    ppt_table.to_excel(writer, sheet_name="PPT_Summary", index=False)
    theme_counts.to_excel(writer, sheet_name="All_Themes", index=False)
    df.to_excel(writer, sheet_name="Review_Details", index=False)
    validation_sample.to_excel(
        writer, sheet_name="Validation_Sample", index=False
    )

print("Excel tersimpan:", output_file)

if "Previous_Sentiment" in df.columns:
    changed = (
        df["Previous_Sentiment"].astype(str).str.capitalize()
        .ne(df["Text_Sentiment"])
        & df["Text_Sentiment"].ne("Not analyzed")
    )
    print("Label berbeda dari hasil sebelumnya:", changed.sum())

Jumlah review: 23,394
Teks yang berubah: 4657


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 547.24it/s]


Model labels: {0: 'positive', 1: 'neutral', 2: 'negative'}


Analyzing sentiment: 100%|██████████| 1257/1257 [12:17<00:00,  1.71it/s]


,Competitor,Reviews Collected,Texts Analyzed,Avg Review Rating,Positive %,Neutral %,Negative %,Top Complaint Theme,Theme Mentions,Status
0,Ajaib,15557,15557,4.22,63.9,9.5,26.7,App stability / speed,657,Normalized RoBERTa — not manually validated
1,IPOT,1631,1631,3.31,41.4,7.4,51.2,App stability / speed,212,Normalized RoBERTa — not manually validated
2,KISI (KOINS Mobile),19,19,2.84,21.1,10.5,68.4,App stability / speed,8,Normalized RoBERTa — not manually validated
3,KISI (iKISI),131,131,4.18,65.6,11.5,22.9,Registration / verification,7,Normalized RoBERTa — not manually validated
4,Mirae Asset (Neo HOTS),116,116,1.97,12.1,10.3,77.6,Login / account access,39,Normalized RoBERTa — not manually validated
5,Stockbit,5940,5940,3.91,49.1,11.9,39.1,App stability / speed,753,Normalized RoBERTa — not manually validated


Excel tersimpan: D:\KISI_Competitor_Analysis\KISI_RoBERTa_Normalized_20261006_115908.xlsx
Label berbeda dari hasil sebelumnya: 597


In [15]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\KISI_Competitor_Analysis")
file = folder / "google_play_roberta_normalized_detail.csv"

df = pd.read_csv(file, keep_default_na=False)

apps = [
    "KISI (iKISI)",
    "KISI (KOINS Mobile)",
    "Stockbit",
    "Ajaib",
]

selected = df[
    df["Competitor"].isin(apps)
    & df["Review_Text"].astype(str).str.strip().ne("")
].drop_duplicates(["Competitor", "Review_Text"])

samples = []

for app in apps:
    for sentiment in ["Positive", "Negative"]:
        group = selected[
            selected["Competitor"].eq(app)
            & selected["Text_Sentiment"].eq(sentiment)
        ]

        if not group.empty:
            samples.append(
                group.sample(n=min(2, len(group)), random_state=42)
            )

if not samples:
    raise ValueError("Tidak ada review yang sesuai ditemukan.")

result = pd.concat(samples, ignore_index=True)

columns = ["Competitor", "Text_Sentiment", "Review_Text"]
if "score" in result.columns:
    columns.append("score")

result = result[columns].rename(columns={
    "Competitor": "App",
    "Text_Sentiment": "RoBERTa Label",
    "Review_Text": "Review",
    "score": "Stars",
})

pd.set_option("display.max_colwidth", None)
display(result)

,App,RoBERTa Label,Review,Stars
0,KISI (iKISI),Positive,aplikasi nya bagus banget dan mudah di pahami untuk pemula,5
1,KISI (iKISI),Positive,"Semua OK dari segi tampilan, performa dan layanan. 👍",5
2,KISI (iKISI),Negative,trading balance gak bisa ditarik..ada saldonya tapi di menu penarikan saldo tersedia selalu 0..payah,1
3,KISI (iKISI),Negative,daftar ya susah gimna ij cara daftar ia,2
4,KISI (KOINS Mobile),Positive,apk paling okk,5
5,KISI (KOINS Mobile),Positive,"belajar saham dari kisi semarang dan ngerti saham. dari tidak tau menjadi tahu, belajar saham untuk pemula bersama kisi semarang dengan ngerti saham",5
6,KISI (KOINS Mobile),Negative,"kenapa withdrawl ke rekening pribadi lama sekali?? padahal dari jam 08:00 wib, sampai jam 14:30 belum masuk, lagi perlu uang",3
7,KISI (KOINS Mobile),Negative,skrg jd suka eror,1
8,Stockbit,Positive,"bagus fiturnya banyak dan mudah dimengerti juga,tapi kendala nya kalo mau wd suka lama banget nunggu dari pagi blom juga masuk",4
9,Stockbit,Positive,Sangat bagus untuk investasi terutama bagi pemula,5


In [16]:
from pathlib import Path
from io import StringIO
import pandas as pd

file = Path(r"D:\KISI_Competitor_Analysis\multiTimeline (9).csv")

if not file.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {file}")

# Cari baris header karena CSV Trends punya keterangan di awal
lines = file.read_text(encoding="utf-8-sig").splitlines()

header_row = next(
    (
        i for i, line in enumerate(lines)
        if line.split(",")[0].strip().strip('"')
        in ["Week", "Day", "Month", "Minggu", "Hari", "Bulan"]
    ),
    None,
)

if header_row is None:
    print("\n".join(lines[:10]))
    raise ValueError("Header belum dikenali. Kirim output di atas.")

df_trends = pd.read_csv(
    StringIO("\n".join(lines[header_row:]))
)

print("Kolom:", df_trends.columns.tolist())
print("Jumlah baris:", len(df_trends))
display(df_trends.head())

Kolom: ['Week', 'ikisi: (Indonesia)', 'stockbit: (Indonesia)', 'mirae: (Indonesia)', 'ajaib: (Indonesia)', 'IPOT: (Indonesia)']
Jumlah baris: 53


,Week,ikisi: (Indonesia),stockbit: (Indonesia),mirae: (Indonesia),ajaib: (Indonesia),IPOT: (Indonesia)
0,2025-10-05,0,60,9,56,14
1,2025-10-12,0,56,7,55,13
2,2025-10-19,0,49,7,49,12
3,2025-10-26,0,49,8,51,12
4,2025-11-02,0,56,10,51,12
